# ACE-Step Captioner -> Suno prompt (Colab T4)

Runs the pipeline from `AGENTS.md`: **audio -> ACE-Step captioner -> prose caption -> lexicon tag extractor -> Suno prompt string**.

Runtime: Colab **T4 GPU**. The captioner runs via **HuggingFace transformers in 4-bit (nf4)** — no llama.cpp build required. Run all cells top to bottom; downloads are cached and idempotent.

> T4 notes: fp16 compute (no native bf16), SDPA attention (no FlashAttention).

In [ ]:
# 0. Confirm GPU + get the repo (skip if already here)
!nvidia-smi -L
import os
if not os.path.exists('/content/music_captioning_oct_2026'):
    !git clone <YOUR_REPO_URL> /content/music_captioning_oct_2026
%cd /content/music_captioning_oct_2026

In [ ]:
# 1. Python deps
!pip -q install -r requirements.txt

In [ ]:
# 2. Download the 4-bit captioner (~6.4 GB), plus spk_dict.pt from the base repo. Idempotent.
from huggingface_hub import snapshot_download, hf_hub_download
snapshot_download("Urabewe/Ace-Step-Captioner-4bit", local_dir="models/Ace-Step-Captioner-4bit",
                  allow_patterns=["*.json", "*.safetensors", "*.jinja", "*.txt", "*.model"])
hf_hub_download("ACE-Step/acestep-captioner", "spk_dict.pt", local_dir="models/Ace-Step-Captioner-4bit")
print("model ready")

In [ ]:
# 3. Provide an audio file (upload, or point at a Drive path).
AUDIO = "/content/elisa_maktooba_leek.mp3"
import os
if not os.path.exists(AUDIO):
    from google.colab import files
    up = files.upload()
    AUDIO = "/content/" + next(iter(up))
print(AUDIO)

In [ ]:
# 4. Run the pipeline end to end (captions 60 s chunks, merges tags).
import json
from src.captioner_hf import HFCaptioner, caption_and_tag

from pathlib import Path
cap = HFCaptioner(model_dir="models/Ace-Step-Captioner-4bit", chunk_seconds=60.0)
result = caption_and_tag(cap, AUDIO)

out = Path("outputs") / (Path(AUDIO).stem + ".json")
out.parent.mkdir(parents=True, exist_ok=True)
out.write_text(json.dumps(result, ensure_ascii=False, indent=2))
print("Saved:", out, "\n")
print("Caption:\n", result["caption"], "\n")
print("Suno prompt:\n", result["suno_prompt"], "\n")
print(json.dumps({k: result[k] for k in ("instruments", "vocals", "production")}, indent=2))

In [ ]:
# 5. CPU-only tag-extraction sanity check (no GPU needed)
!python -m pytest tests/ -q

## Appendix: llama.cpp GGUF backend (alternative)

The pinned GGUF runtime is still supported via `src/captioner.py` and `scripts/build_llamacpp.sh`. It needs a CUDA build of the patched fork (no Linux CUDA prebuilt exists) and the two GGUF files in `models/`. See `docs/RESULTS.md`.